# Task B1 — Image Project: Vehicle Classifier (Teachable Machine → Python)


## B1.3 · Setup, export verification


In [6]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # Teachable Machine .h5 files need legacy Keras (pip install tf_keras)
import time, json, csv, threading
from pathlib import Path
import numpy as np, cv2
import tensorflow as tf
try:
    import tf_keras as keras      # Teachable Machine .h5 files need legacy Keras 2 -> pip install tf_keras
except ImportError as e:
    raise ImportError("Run:  pip install tf_keras   (same minor version as TensorFlow), then restart the kernel") from e
np.set_printoptions(suppress=True)
print("TensorFlow", tf.__version__, "| tf_keras", keras.__version__, "| OpenCV", cv2.__version__)

HERE = Path.cwd()
MODEL_PATH, LABELS_PATH = HERE / "models" / "keras_model.h5", HERE / "models" / "labels.txt"
OUT = HERE / "output"; OUT.mkdir(exist_ok=True)
assert MODEL_PATH.exists() and LABELS_PATH.exists(), "Run this notebook from the image/ folder"

TensorFlow 2.21.0 | tf_keras 2.21.0 | OpenCV 5.0.0


In [7]:
model = keras.models.load_model(MODEL_PATH, compile=False)
labels = [l.strip().split(" ", 1)[1] for l in open(LABELS_PATH) if l.strip()]
print(f"{len(labels)} classes:", labels)
assert model.output_shape[-1] == len(labels), "labels.txt does not match model output size"
print("input :", model.input_shape, "\noutput:", model.output_shape)

20 classes: ['Airplane', 'Ambulance', 'Bicycle', 'Boat', 'Bus', 'Car', 'Fire Truck', 'Helicopter', 'Hovercraft', 'Jet Ski', 'Kayak', 'Motorcycle', 'Rickshaw', 'Scooter', 'Segway', 'Skateboard', 'Tractor', 'Truck', 'Unicycle', 'Van']
input : (None, 224, 224, 3) 
output: (None, 20)


## B4.4 · Model introspection (image model)
Teachable Machine builds a **MobileNet-style feature extractor** followed by a small **Dense head**; the head is what you trained on your own classes.

In [8]:
model.summary(expand_nested=False)

def leaf_layers(m):
    for l in m.layers:
        yield from leaf_layers(l) if hasattr(l, "layers") else [l]

n_params = lambda ws: sum(int(np.prod(w.shape)) for w in ws)
print("\nTop-level blocks :", [l.name for l in model.layers])
print("Total leaf layers:", len(list(leaf_layers(model))))
print(f"Total params     : {model.count_params():,}")
print(f"Trainable params : {n_params(model.trainable_weights):,}")
for blk in model.layers:
    print(f"  {blk.name:14s} params={n_params(blk.weights):>9,}  trainable={n_params(blk.trainable_weights):>9,}")
head_layers = [l for l in leaf_layers(model) if l.__class__.__name__ == "Dense"]
print("Dense head layers:", [(l.name, l.units) for l in head_layers])
print(f"Head parameters (the classes you trained): {n_params([w for l in head_layers for w in l.weights]):,}")

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 20)                130100    
                                                                 
Total params: 540308 (2.06 MB)
Trainable params: 526228 (2.01 MB)
Non-trainable params: 14080 (55.00 KB)
_________________________________________________________________

Top-level blocks : ['sequential_1', 'sequential_3']
Total leaf layers: 158
Total params     : 540,308
Trainable params : 526,228
  sequential_1   params=  410,208  trainable=  396,128
  sequential_3   params=  130,100  trainable=  130,100
Dense head layers: [('dense_Dense1', 100), ('dense_Dense2', 20)]
Head parameters (the classes you trained): 130,100


## B1.4 · Preprocessing + still-image prediction
Frames are **center-cropped to a square** (like the Teachable Machine preview), resized to **224×224**, and scaled to **[-1, 1]** via `img / 127.5 - 1` — the same transform used in training.

In [9]:
def preprocess_bgr(frame_bgr):
    h, w = frame_bgr.shape[:2]; s = min(h, w)
    y0, x0 = (h - s) // 2, (w - s) // 2
    sq = frame_bgr[y0:y0 + s, x0:x0 + s]
    rgb = cv2.cvtColor(cv2.resize(sq, (224, 224), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
    return (rgb.astype(np.float32) / 127.5 - 1.0)[None]                   # shape (1, 224, 224, 3)

def classify_bgr(frame_bgr):
    probs = model.predict(preprocess_bgr(frame_bgr), verbose=0)[0]
    idx = int(np.argmax(probs)); return idx, float(probs[idx]), probs

# range check
x = preprocess_bgr(np.random.randint(0, 256, (480, 640, 3), np.uint8))
print("tensor", x.shape, "min", x.min().round(3), "max", x.max().round(3))

TEST_IMAGE = None   # e.g. "test_images/bus.jpg"
if TEST_IMAGE:
    idx, conf, probs = classify_bgr(cv2.imread(TEST_IMAGE))
    print(f"Prediction: {labels[idx]}  ({conf:.2%})")
    for i in np.argsort(probs)[::-1][:3]: print(f"  {labels[i]:12s} {probs[i]:.2%}")

tensor (1, 224, 224, 3) min -0.875 max 0.827


### Optional: measure accuracy on the Kaggle dataset
Point `DATASET_DIR` to the folder that has one sub-folder per class (folder names must equal the labels, case-insensitive).

In [10]:
DATASET_DIR = None   # e.g. r"C:/data/vehicle_data"
if DATASET_DIR:
    lab = {l.lower(): i for i, l in enumerate(labels)}; ok = n = 0
    for cls in sorted(Path(DATASET_DIR).iterdir()):
        if cls.name.lower() not in lab: continue
        for f in list(cls.glob("*.*"))[:30]:
            im = cv2.imread(str(f))
            if im is None: continue
            n += 1; ok += classify_bgr(im)[0] == lab[cls.name.lower()]
    print(f"Accuracy on {n} images: {ok / max(n, 1):.2%}")

## B1.4 / B1.5 · Webcam deployment + useful extension

```
APPLICATION: Vehicle Logger
 - What it does : classifies each webcam frame into one of 20 vehicle classes and, every LOG_EVERY seconds,
                  appends  timestamp, class, confidence  to output/b1_vehicle_log.csv (only if confidence >= MIN_CONF).
 - How to use   : show a toy vehicle / a picture of a vehicle to the camera.
                  q = quit   s = save screenshot   (the last frame is saved to output/b1_image_result.jpg on exit)
 - Recognises   : Airplane, Ambulance, Bicycle, Boat, Bus, Car, Fire Truck, Helicopter, Hovercraft, Jet Ski,
                  Kayak, Motorcycle, Rickshaw, Scooter, Segway, Skateboard, Tractor, Truck, Unicycle, Van
```
`webcam_loop()` is the plain deployment script (B1.4); passing `log_csv=` turns it into the logger (B1.5).

In [11]:
def draw_overlay(frame, probs, fps=None):
    out = frame.copy(); top = np.argsort(probs)[::-1][:3]
    cv2.rectangle(out, (0, 0), (330, 30 + 28 * 3), (0, 0, 0), -1)
    for r, i in enumerate(top):
        y = 24 + r * 28; col = (0, 255, 0) if r == 0 else (200, 200, 200)
        cv2.putText(out, f"{labels[i]} {probs[i]:.0%}", (8, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2, cv2.LINE_AA)
        cv2.rectangle(out, (190, y - 14), (190 + int(130 * probs[i]), y + 2), col, -1)      # confidence bar
    if fps: cv2.putText(out, f"{fps:.0f} FPS", (out.shape[1] - 90, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    return out

def webcam_loop(camera=0, mirror=True, log_csv=None, log_every=5.0, min_conf=0.6, max_seconds=300):
    cap = cv2.VideoCapture(camera)
    if not cap.isOpened(): raise RuntimeError("Cannot open webcam - close other apps using it or change camera=")
    if log_csv and not Path(log_csv).exists():
        with open(log_csv, "w", newline="") as f: csv.writer(f).writerow(["timestamp", "class", "confidence"])
    t0 = last_log = time.time(); last = None
    try:
        while time.time() - t0 < max_seconds:
            ok, frame = cap.read()
            if not ok: break
            if mirror: frame = cv2.flip(frame, 1)
            t = time.time(); idx, conf, probs = classify_bgr(frame)
            last = draw_overlay(frame, probs, 1 / max(time.time() - t, 1e-6))
            if log_csv and time.time() - last_log >= log_every and conf >= min_conf:
                with open(log_csv, "a", newline="") as f:
                    csv.writer(f).writerow([time.strftime("%Y-%m-%d %H:%M:%S"), labels[idx], f"{conf:.3f}"])
                last_log = time.time(); print("logged:", labels[idx], f"{conf:.2%}")
            cv2.imshow("B1 Image Classifier  (q = quit, s = screenshot)", last)
            k = cv2.waitKey(1) & 0xFF
            if k == ord("q"): break
            if k == ord("s"): cv2.imwrite(str(OUT / f"screenshot_{int(time.time())}.jpg"), last)
    finally:
        cap.release(); cv2.destroyAllWindows(); cv2.waitKey(1)
        if last is not None: cv2.imwrite(str(OUT / "b1_image_result.jpg"), last); print("saved", OUT / "b1_image_result.jpg")

### Run B1.4 — plain webcam classifier (press **q** to stop)

In [ ]:
RUN_WEBCAM = True
if RUN_WEBCAM: webcam_loop()

### Run B1.5 — Vehicle Logger (logs every 5 s while a vehicle is recognised)

In [ ]:
LOG_FILE = OUT / "b1_vehicle_log.csv"
if RUN_WEBCAM: webcam_loop(log_csv=LOG_FILE, log_every=5.0, min_conf=0.6)

In [ ]:
import pandas as pd
if LOG_FILE.exists() and LOG_FILE.stat().st_size > 0:
    log = pd.read_csv(LOG_FILE); display(log.tail(10))
    if len(log): log["class"].value_counts().plot(kind="bar", title="Vehicles logged")